In [1]:
import pandas as pd
import numpy as np
import optuna
import json
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('playground-series-s6e9/train.csv')
test = pd.read_csv('playground-series-s6e9/test.csv')
sample_sub = pd.read_csv('playground-series-s6e9/sample_submission.csv')

# Preprocessing
target = 'Will_Buy_EV'
features = [c for c in train.columns if c not in ['id', target]]

cat_cols = train[features].select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

train[target] = train[target].map({'Yes': 1, 'No': 0})

X = train[features]
y = train[target]
X_test = test[features]

num_trials = 50
n_splits = 5


In [2]:
import lightgbm as lgb

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'scale_pos_weight': float(np.sum(y == 0) / np.sum(y == 1)),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'verbose': -1
    }
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc_scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        callbacks = [lgb.early_stopping(50, verbose=False)]
        model = lgb.train(params, train_data, num_boost_round=1000, valid_sets=[val_data], callbacks=callbacks)
        preds = model.predict(X_val)
        auc = roc_auc_score(y_val, preds)
        auc_scores.append(auc)
        
    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=num_trials)
print("Best trial:", study.best_trial.params)

best_params = study.best_trial.params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'
best_params['boosting_type'] = 'gbdt'
best_params['scale_pos_weight'] = float(np.sum(y == 0) / np.sum(y == 1))
best_params['verbose'] = -1

with open('lightgbm_best_params.json', 'w') as f:
    json.dump(best_params, f)


[I 2026-09-10 19:33:20,303] A new study created in memory with name: no-name-68e96ca0-b15b-4886-9ce8-99fa08571a21


[I 2026-09-10 21:29:17,090] Trial 0 finished with value: 0.9415328111125074 and parameters: {'learning_rate': 0.011567283616889966, 'num_leaves': 53, 'max_depth': 10, 'min_child_samples': 75, 'subsample': 0.9635225263219961, 'colsample_bytree': 0.8295508768795823, 'reg_alpha': 0.0010191512784865106, 'reg_lambda': 9.386244670582509e-08}. Best is trial 0 with value: 0.9415328111125074.


[I 2026-09-10 21:48:41,869] Trial 1 finished with value: 0.9408103456478365 and parameters: {'learning_rate': 0.017829840400691577, 'num_leaves': 109, 'max_depth': 3, 'min_child_samples': 28, 'subsample': 0.6370909187460191, 'colsample_bytree': 0.8779873511162875, 'reg_alpha': 0.08328338493715338, 'reg_lambda': 2.1757708904883895e-06}. Best is trial 0 with value: 0.9415328111125074.


[I 2026-09-10 23:51:35,672] Trial 2 finished with value: 0.9417614910586785 and parameters: {'learning_rate': 0.03341287728893038, 'num_leaves': 113, 'max_depth': 10, 'min_child_samples': 92, 'subsample': 0.9599623786399203, 'colsample_bytree': 0.6033018417927529, 'reg_alpha': 0.001236114921884228, 'reg_lambda': 0.00035094890753396016}. Best is trial 2 with value: 0.9417614910586785.


[I 2026-09-11 00:14:50,171] Trial 3 finished with value: 0.9372464869477792 and parameters: {'learning_rate': 0.003061782669293762, 'num_leaves': 67, 'max_depth': 4, 'min_child_samples': 67, 'subsample': 0.9982353555496257, 'colsample_bytree': 0.9804421527806573, 'reg_alpha': 1.1487140762047943, 'reg_lambda': 0.2012182678532658}. Best is trial 2 with value: 0.9417614910586785.


...

[I 2026-09-11 11:28:32,049] Trial 46 finished with value: 0.9418870108853739 and parameters: {'learning_rate': 0.035455823934491364, 'num_leaves': 57, 'max_depth': 9, 'min_child_samples': 60, 'subsample': 0.7265878533393113, 'colsample_bytree': 0.5338620462379601, 'reg_alpha': 2.0705990576803532e-05, 'reg_lambda': 0.9082622852834771}. Best is trial 33 with value: 0.9419311364746891.


[I 2026-09-11 11:29:50,469] Trial 47 finished with value: 0.9419038032350585 and parameters: {'learning_rate': 0.06525196593400452, 'num_leaves': 38, 'max_depth': 9, 'min_child_samples': 64, 'subsample': 0.7498577013278757, 'colsample_bytree': 0.5310740493591587, 'reg_alpha': 1.6495842654440346e-05, 'reg_lambda': 2.9099057511076496}. Best is trial 33 with value: 0.9419311364746891.


[I 2026-09-11 11:31:25,113] Trial 48 finished with value: 0.9419062613871179 and parameters: {'learning_rate': 0.04240840323137119, 'num_leaves': 58, 'max_depth': 10, 'min_child_samples': 49, 'subsample': 0.7455697886384707, 'colsample_bytree': 0.5278773937661024, 'reg_alpha': 1.4563182806873662e-07, 'reg_lambda': 3.396298634398207}. Best is trial 33 with value: 0.9419311364746891.


[I 2026-09-11 11:33:15,622] Trial 49 finished with value: 0.9418925658592399 and parameters: {'learning_rate': 0.03624340256039458, 'num_leaves': 40, 'max_depth': 9, 'min_child_samples': 81, 'subsample': 0.7915019318219454, 'colsample_bytree': 0.5391316272201755, 'reg_alpha': 1.7877978744525775e-06, 'reg_lambda': 0.7896569738771815}. Best is trial 33 with value: 0.9419311364746891.


Best trial: {'learning_rate': 0.04949343841645206, 'num_leaves': 29, 'max_depth': 10, 'min_child_samples': 70, 'subsample': 0.8407490348740534, 'colsample_bytree': 0.5191809493711131, 'reg_alpha': 2.0215203272011218e-05, 'reg_lambda': 5.410864199001933}


In [3]:
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_preds_df = pd.DataFrame({'id': test['id']})

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    callbacks = [lgb.early_stopping(50, verbose=False)]
    model = lgb.train(best_params, train_data, num_boost_round=2000, valid_sets=[val_data], callbacks=callbacks)
    
    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds
    fold_test_preds = model.predict(X_test)
    test_preds += fold_test_preds / n_splits
    fold_preds_df[f'fold_{fold}'] = fold_test_preds

print("OOF AUC:", roc_auc_score(y, oof_preds))

pd.DataFrame({'id': train['id'], 'Will_Buy_EV': oof_preds}).to_csv('lightgbm_oof_predictions.csv', index=False)
fold_preds_df.to_csv('lightgbm_fold_predictions.csv', index=False)

sample_sub['Will_Buy_EV'] = test_preds
sample_sub.to_csv('submission_lightgbm.csv', index=False)


OOF AUC: 0.9419236200246852
